# Day 59 — CNN Foundations
- Explain the visual-cortex inspiration behind CNNs.
- Describe filters, feature maps, stride, padding, and convolution.
- Implement convolutional and pooling layers with TensorFlow/Keras.
- Calculate output shapes and estimate activation memory.
- Compare max pooling and average pooling.

## 1. Why convolutional neural networks?
Images have spatial structure: nearby pixels often relate to one another. A dense layer connects every input pixel to every neuron, creating many parameters and ignoring the local layout.

CNNs use **local receptive fields** and **shared filters**:
- **Local receptive field:** a neuron looks at a small patch of the image.
- **Filter/kernel:** a small array of learned weights slides across the image.
- **Feature map:** the filter's response at each spatial location.
- **Hierarchical features:** early layers may detect edges; deeper layers can combine them into textures, parts, and object-level patterns.

CNNs are loosely inspired by visual processing in the brain, where neurons respond to patterns in restricted parts of the visual field. This is an inspiration, not a literal copy of the brain.

## 2. Convolution intuition
A 3×3 filter is applied to each 3×3 patch. Multiply corresponding values, sum them, optionally add a bias, and write the result into the output feature map.

For a 2-D input of size \(n\times n\), filter size \(f\), stride \(s\), and no padding, the output size per dimension is:

\[
\left\lfloor\frac{n-f}{s}\right\rfloor+1
\]

With `"same"` padding, TensorFlow generally pads so output spatial dimensions are \(\lceil n/s\rceil\). For stride 1, output height and width stay the same.

### A tiny hand-worked example

In [1]:
import numpy as np

image = np.array([
    [1, 2, 0, 1],
    [0, 1, 3, 1],
    [2, 2, 1, 0],
    [1, 0, 2, 2]
], dtype=np.float32)

kernel = np.array([
    [1, 0],
    [0, -1]
], dtype=np.float32)

def valid_cross_correlation(image, kernel, stride=1):
    h, w = image.shape
    kh, kw = kernel.shape
    out_h = (h - kh) // stride + 1
    out_w = (w - kw) // stride + 1
    output = np.zeros((out_h, out_w), dtype=np.float32)
    for y in range(out_h):
        for x in range(out_w):
            patch = image[y*stride:y*stride+kh, x*stride:x*stride+kw]
            output[y, x] = np.sum(patch * kernel)
    return output

print("Input image:\n", image)
print("Kernel:\n", kernel)
print("Output feature map:\n", valid_cross_correlation(image, kernel))

Input image:
 [[1. 2. 0. 1.]
 [0. 1. 3. 1.]
 [2. 2. 1. 0.]
 [1. 0. 2. 2.]]
Kernel:
 [[ 1.  0.]
 [ 0. -1.]]
Output feature map:
 [[ 0. -1. -1.]
 [-2.  0.  3.]
 [ 2.  0. -1.]]


**Note:** Many deep-learning libraries implement cross-correlation (the filter is not flipped), although the layer is commonly called convolution. During training, the filter weights are learned from data.

## 3. Stride, padding, channels, and output shape
- **Stride 1:** move the filter one pixel each step; finer spatial output.
- **Stride 2:** move two pixels each step; smaller output, less computation.
- **`padding="valid"`:** no padding; output shrinks.
- **`padding="same"`:** pad to preserve spatial dimensions when stride is 1.
- **Input channels:** an RGB image has 3 channels. A convolutional filter spans all input channels.
- **Number of filters:** determines the output channel count.

For input shape `(batch, height, width, channels)`, Keras `Conv2D` output is `(batch, out_height, out_width, filters)`.

### Install/import TensorFlow and check your version

In [2]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow:", tf.__version__)

TensorFlow: 2.21.0


### Conv2D example

In [3]:
# Batch of 8 RGB images, each 32×32
x = tf.random.normal((8, 32, 32, 3))

conv_valid = layers.Conv2D(filters=16, kernel_size=3, strides=1,
                           padding="valid", activation="relu")
y_valid = conv_valid(x)

conv_same = layers.Conv2D(filters=16, kernel_size=3, strides=1,
                          padding="same", activation="relu")
y_same = conv_same(x)

print("Input:", x.shape)
print("valid output:", y_valid.shape)  # (8, 30, 30, 16)
print("same output: ", y_same.shape)   # (8, 32, 32, 16)
print("Conv parameters:", conv_valid.count_params())

Input: (8, 32, 32, 3)
valid output: (8, 30, 30, 16)
same output:  (8, 32, 32, 16)
Conv parameters: 448


### Parameter-count formula
For a `k_h × k_w` kernel, `C_in` input channels, `C_out` filters, and a bias per filter:

\[
(k_h k_w C_{in} + 1) C_{out}
\]

For a 3×3 convolution with 3 input channels and 16 filters: \((3\cdot3\cdot3+1)\cdot16=448\) parameters. The number of parameters does **not** increase with image height and width; the same filters are reused across locations.

In [4]:
kh, kw, in_channels, filters = 3, 3, 3, 16
manual_params = (kh * kw * in_channels + 1) * filters
print("Manual parameter count:", manual_params)

Manual parameter count: 448


## 4. Pooling layers
Pooling downsamples feature maps and can reduce compute and activation memory in later layers.

- **Max pooling:** keeps the largest value in each local window. Often preserves strong detected features.
- **Average pooling:** keeps the average value. Produces smoother summaries.
- Pooling layers usually have **no trainable parameters**.
- Pooling is applied independently to each channel; it does not mix channels.
- Pooling can lose precise spatial information, so use it thoughtfully.

For a 2×2 window and stride 2, a 4×4 feature map becomes 2×2.

### Compare max pooling and average pooling

In [5]:
feature_map = tf.constant([[
    [[1.], [3.], [2.], [4.]],
    [[5.], [6.], [1.], [2.]],
    [[0.], [2.], [7.], [8.]],
    [[3.], [1.], [4.], [6.]]
]])  # shape: (batch=1, height=4, width=4, channels=1)

max_pool = layers.MaxPooling2D(pool_size=2, strides=2)
avg_pool = layers.AveragePooling2D(pool_size=2, strides=2)

print("Input shape:", feature_map.shape)
print("Max pooled:\n", max_pool(feature_map).numpy()[0, :, :, 0])
print("Average pooled:\n", avg_pool(feature_map).numpy()[0, :, :, 0])

Input shape: (1, 4, 4, 1)
Max pooled:
 [[6. 4.]
 [3. 8.]]
Average pooled:
 [[3.75 2.25]
 [1.5  6.25]]


### Pooling with `padding='same'`

In [6]:
odd_map = tf.random.normal((1, 5, 5, 8))
pool_valid = layers.MaxPooling2D(pool_size=2, strides=2, padding="valid")
pool_same = layers.MaxPooling2D(pool_size=2, strides=2, padding="same")

print("Input:", odd_map.shape)
print("valid:", pool_valid(odd_map).shape)
print("same: ", pool_same(odd_map).shape)

Input: (1, 5, 5, 8)
valid: (1, 2, 2, 8)
same:  (1, 3, 3, 8)


## 5. Memory requirements: why feature maps matter
A CNN stores intermediate **activations** so backpropagation can compute gradients. Training memory also includes gradients, model parameters, optimizer state, temporary workspaces, and framework overhead.

A rough activation-memory estimate is:

\[
B\times H\times W\times C\times\text{bytes per value}
\]

For float32, each value uses 4 bytes. This estimate is for one tensor only—not the full training memory.

In [7]:
def activation_memory_mib(batch, height, width, channels, bytes_per_value=4):
    total_bytes = batch * height * width * channels * bytes_per_value
    return total_bytes / (1024 ** 2)

for shape in [(32, 224, 224, 64), (32, 112, 112, 128), (32, 56, 56, 256)]:
    print(shape, "->", round(activation_memory_mib(*shape), 2), "MiB per tensor")

(32, 224, 224, 64) -> 392.0 MiB per tensor
(32, 112, 112, 128) -> 196.0 MiB per tensor
(32, 56, 56, 256) -> 98.0 MiB per tensor


### Ways to manage memory
1. Reduce **batch size** first; activation memory scales linearly with it.
2. Downsample earlier when task accuracy allows.
3. Reduce the number of filters or input resolution if appropriate.
4. Use mixed precision on supported hardware (verify numerical stability and device support).
5. Avoid keeping unnecessary intermediate tensors in Python variables.
6. For large models, consider gradient checkpointing/recomputation if your framework/model supports it.

**Important:** Pooling reduces the spatial dimensions of later activations, but it does not automatically erase every earlier activation required for backpropagation.

## 6. Build a small CNN
This model is for shape and layer practice; it is not intended as a state-of-the-art classifier.

In [8]:
model = keras.Sequential([
    keras.Input(shape=(32, 32, 3)),
    layers.Conv2D(32, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(pool_size=2),
    layers.Conv2D(64, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(pool_size=2),
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dense(10, activation="softmax")
])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)               │ (None, 32, 32, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 16, 16, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │       262,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 282,250 (1.08 MB)

 Trainable params: 282,250 (1.08 MB)

 Non-trainable params: 0 (0.00 B)